In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="kenexaihackathon",
    user="postgres",
    password="09092002"
)

# -----------------------------
# LOAD BRONZE DATA
# -----------------------------

meraki = pd.read_sql("SELECT * FROM bronze.meraki_alerts", conn)
auvik = pd.read_sql("SELECT * FROM bronze.auvik_alerts", conn)
ncentral = pd.read_sql("SELECT * FROM bronze.ncentral_alerts", conn)


# -----------------------------
# NORMALIZATION FUNCTIONS
# -----------------------------

def normalize_severity(sev):

    if sev is None:
        return "LOW"

    sev = str(sev).lower()

    if "critical" in sev or "failed" in sev:
        return "HIGH"

    if "warning" in sev:
        return "MEDIUM"

    return "LOW"


def normalize_event_state(state):

    if state is None:
        return "triggered"

    state = str(state).lower()

    if state in ["ok","normal","cleared","resolved"]:
        return "resolved"

    return "triggered"


# -----------------------------
# PROCESS MERAKI
# -----------------------------

meraki_clean = pd.DataFrame({

    "source_system": "meraki",

    "alert_id": meraki["alert_id"],

    "correlation_id": meraki["correlation_id"].fillna(meraki["alert_id"]),

    "organization_name": meraki["organization_name"],

    "device_name": meraki["device_name"].str.lower().str.strip(),

    "device_identifier": meraki["device_serial"],

    "service_name": meraki["check_name"],

    "alert_type": meraki["alert_type"],

    "severity": meraki["alert_level"].apply(normalize_severity),

    "event_state": meraki["event_state"].apply(normalize_event_state),

    "event_time": meraki["occurred_at"],

    "synthetic": meraki["synthetic"],

    "ingestion_time": meraki["ingestion_time"]

})


# -----------------------------
# PROCESS AUVIK
# -----------------------------

auvik_clean = pd.DataFrame({

    "source_system": "auvik",

    "alert_id": auvik["alert_id"],

    "correlation_id": auvik["correlation_id"].fillna(auvik["alert_id"]),

    "organization_name": auvik["company_name"],

    "device_name": auvik["entity_name"].str.lower().str.strip(),

    "device_identifier": auvik["entity_id"],

    "service_name": auvik["entity_type"],

    "alert_type": auvik["alert_name"],

    "severity": auvik["alert_severity_string"].apply(normalize_severity),

    "event_state": auvik["event_state"].apply(normalize_event_state),

    "event_time": auvik["event_time"],

    "synthetic": auvik["synthetic"],

    "ingestion_time": auvik["ingestion_time"]

})


# -----------------------------
# PROCESS NCENTRAL
# -----------------------------

ncentral_clean = pd.DataFrame({

    "source_system": "ncentral",

    "alert_id": ncentral["active_notification_trigger_id"],

    "correlation_id": ncentral["correlation_id"].fillna(
        ncentral["active_notification_trigger_id"]
    ),

    "organization_name": ncentral["customer_name"],

    "device_name": ncentral["device_name"].str.lower().str.strip(),

    "device_identifier": ncentral["device_uri"],

    "service_name": ncentral["affected_service"],

    "alert_type": ncentral["affected_service"],

    "severity": ncentral["qualitative_new_state"].apply(normalize_severity),

    "event_state": ncentral["event_state"].apply(normalize_event_state),

    "event_time": ncentral["time_of_state_change"],

    "synthetic": ncentral["synthetic"],

    "ingestion_time": ncentral["ingestion_time"]

})


# -----------------------------
# COMBINE ALL ALERTS
# -----------------------------

silver_df = pd.concat([
    meraki_clean,
    auvik_clean,
    ncentral_clean
])


# -----------------------------
# REMOVE DUPLICATES
# -----------------------------

silver_df = silver_df.drop_duplicates(
    subset=[
        "correlation_id",
        "device_name",
        "alert_type",
        "event_state",
        "event_time"
    ]
)

# -----------------------------
# LOAD INTO SILVER TABLE
# -----------------------------

cursor = conn.cursor()

for _, row in silver_df.iterrows():

    cursor.execute("""
    INSERT INTO silver.alerts_clean
    (
        source_system,
        alert_id,
        correlation_id,
        organization_name,
        device_name,
        device_identifier,
        service_name,
        alert_type,
        severity,
        event_state,
        event_time,
        synthetic,
        ingestion_time
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """,
    tuple(row)
    )


conn.commit()

cursor.close()
conn.close()

print("Silver preprocessing completed.")


C:\Users\Palak\AppData\Local\Temp\ipykernel_26984\777469743.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  meraki = pd.read_sql("SELECT * FROM bronze.meraki_alerts", conn)
C:\Users\Palak\AppData\Local\Temp\ipykernel_26984\777469743.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  auvik = pd.read_sql("SELECT * FROM bronze.auvik_alerts", conn)
C:\Users\Palak\AppData\Local\Temp\ipykernel_26984\777469743.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ncentral = pd.read_sql("SELECT * FROM bronze.ncentral_alerts",

Silver preprocessing completed.
